<div style="background-image: linear-gradient(to right, #2e5a88, #1e1e1e); color: white; padding: 12px 25px; border-radius: 10px; margin-top: 25px; margin-bottom: 15px;">
    <h1 style="color: white; margin: 0; font-size: 2.0em; font-family: 'Consolas', monospace;">
        00 · ETL Pipeline
    </h1>
    <p style="color: #d1d1d1; margin: 5px 0 0 0; font-size: 1.1em;">Residential Power Consumption Audit · Sceaux Dataset</p>
    <p style="color: #aaaaaa; margin: 4px 0 0 0; font-size: 0.95em;">Aitor — Industrial Electrical Engineer &amp; Data Engineer</p>
</div>

## Alcance de este notebook

Este notebook cubre la capa ETL completa compartida por los cuatro notebooks de hipótesis:

1. **Ingesta de datos** — carga de dataset bruto y validación de esquema  
2. **Data wrangling** — tratamiento de nulos, conversión de tipos, reconstrucción del timestamp  
3. **Feature engineering** — dimensiones temporales (`Hour`, `Is_Weekend`)  
4. **Capa de abstracción SQL** — registro de la vista temporal `power_data`  
5. **Persistencia** — escritura del dataset procesado en Parquet (almacenamiento columnar)

Una vez ejecutado correctamente este notebook, los cuatro notebooks de hipótesis leen desde Parquet directamente — sin re-ingesta ni re-limpieza.

In [1]:
# ==============================================================================
# VERIFICACIÓN DEL DATASET (ejecutar antes de cualquier celda de Spark)
# ==============================================================================
# Este bloque comprueba si el dataset existe. Si no, intenta descargarlo
# automáticamente desde el repositorio oficial de UCI ML Repository.
# Si la descarga falla, muestra instrucciones claras para hacerlo a mano.
# ==============================================================================

import sys
import os

# Añadir la raíz del proyecto al path para importar módulos locales
sys.path.insert(0, os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())

from src.dataset_loader import verify_dataset
from utils.spark_session import resolve_project_root

PROJECT_ROOT = resolve_project_root()
RUTA_DATASET = os.path.join(PROJECT_ROOT, "data_storage", "source", "household_power_consumption.txt")

# --- VERIFICACIÓN ---
dataset_ok = verify_dataset(RUTA_DATASET)

if not dataset_ok:
    raise FileNotFoundError(
        "\n❌ Ejecución detenida: el dataset no está disponible.\n"
        "   Sigue las instrucciones anteriores para obtenerlo manualmente."
    )

------------------------------------------------------------
✅ DATASET ENCONTRADO
   Ruta    : /home/aitor/Documentos/Residential-Power-Audit-Spark-Pipeline/data_storage/source/household_power_consumption.txt
   Tamaño  : 126.8 MB
------------------------------------------------------------


---
## 1. Inicialización del motor Spark

In [2]:
import warnings
import sys
import os

warnings.filterwarnings("ignore")

# --- Permite importar módulos compartidos desde el directorio notebooks/ ---
sys.path.insert(0, os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd())

from utils.spark_session import get_spark, resolve_project_root

spark = get_spark(app_name="ETL_Pipeline")
PROJECT_ROOT = resolve_project_root()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/16 11:31:27 WARN Utils: Your hostname, medion, resolves to a loopback address: 127.0.1.1; using 192.168.1.113 instead (on interface wlo1)
26/05/16 11:31:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/16 11:31:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/05/16 11:31:30 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


-------------------------------------------------------
⚡ SPARK ENGINE READY  |  app: ETL_Pipeline


   Driver : 16g
   Executor: 8g
   Log level: ERROR
-------------------------------------------------------


---
## 2. Ingesta de datos y validación de esquema

### 2.1 Carga de datos brutos

El dataset original usa `;` como separador y `?` como marcador de nulo.
`inferSchema` se establece en `False` intencionalmente — todas las columnas numéricas llegan como cadenas y se convierten explícitamente en 2.2 para garantizar la integridad de tipos.

In [3]:
# ==============================================================================
# 2.1 DATA INGESTION & SCHEMA VALIDATION
# ==============================================================================
import os

# --- RESOLUCIÓN DE RUTA ---
path = os.path.join(PROJECT_ROOT, "data_storage", "source", "household_power_consumption.txt")


# --- CARGA DE DATOS BRUTOS ---
df_raw = spark.read.options(
    header="True",
    sep=";",
    inferSchema="True",
    naStrings="?"
).csv(path)

# --- AUDITORÍA DE DATOS ---
print("-" * 55)
print("📊 INFORME DE AUDITORÍA DE DATOS")
print("-" * 55)
print(f"✅ Registros cargados : {df_raw.count():,}")
print("\n🔍 Esquema detectado:")
df_raw.printSchema()

print("-" * 55)
print("📋 Muestra — primeros 5 registros:")
print("-" * 55)
display(df_raw.limit(5).toPandas())

-------------------------------------------------------
📊 INFORME DE AUDITORÍA DE DATOS
-------------------------------------------------------


✅ Registros cargados : 2,075,259

🔍 Esquema detectado:
root
 |-- Date: string (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- Global_active_power: string (nullable = true)
 |-- Global_reactive_power: string (nullable = true)
 |-- Voltage: string (nullable = true)
 |-- Global_intensity: string (nullable = true)
 |-- Sub_metering_1: string (nullable = true)
 |-- Sub_metering_2: string (nullable = true)
 |-- Sub_metering_3: double (nullable = true)

-------------------------------------------------------
📋 Muestra — primeros 5 registros:
-------------------------------------------------------


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,2026-05-16 17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.0
1,16/12/2006,2026-05-16 17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.0
2,16/12/2006,2026-05-16 17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.0
3,16/12/2006,2026-05-16 17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.0
4,16/12/2006,2026-05-16 17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.0


---
### 2.2 Data Wrangling y Feature Engineering

Se aplican tres transformaciones en un único DAG de Spark para minimizar los shuffles:

| Paso | Acción | Beneficio posterior |
|------|--------|--------------------|
| **Limpieza** | Reemplazar `?` → `null`, eliminar filas incompletas | Elimina errores de lectura del sensor |
| **Casting** | `String` → `Double` para todas las columnas de medida | Habilita aritmética en Spark SQL |
| **Reconstrucción temporal** | Combinar `Date` + `Time` → `Full_Timestamp` | Habilita análisis de ventana secuencial |
| **Feature engineering** | Extraer `Hour`, `Day_Number`, `Is_Weekend` | Segmentación de carga para H1/H3 |

In [4]:
# ==============================================================================
# 2.2 DATA WRANGLING & FEATURE ENGINEERING
# ==============================================================================
from pyspark.sql.functions import (
    col, concat, lit, to_timestamp, hour, dayofweek, when, date_format
)

# --- PASO 1: TRATAMIENTO DE NULOS ---
df_pre_clean = df_raw.replace("?", None)
df_no_nulls  = df_pre_clean.na.drop()

# --- PASO 2: CONVERSIÓN DE TIPOS ---
df_numeric = (
    df_no_nulls
    .withColumn("Global_active_power",  col("Global_active_power").cast("double"))
    .withColumn("Sub_metering_1",       col("Sub_metering_1").cast("double"))
    .withColumn("Sub_metering_2",       col("Sub_metering_2").cast("double"))
    .withColumn("Sub_metering_3",       col("Sub_metering_3").cast("double"))
)

# --- PASO 3: TIMESTAMP + VARIABLES TEMPORALES ---
df_final = (
    df_numeric
    .withColumn(
        "Full_Timestamp",
        to_timestamp(
            concat(col("Date"), lit(" "), date_format(col("Time"), "HH:mm:ss")),
            "d/M/yyyy HH:mm:ss"
        )
    )
    .withColumn("Hour",       hour(col("Full_Timestamp")))
    .withColumn("Day_Number", dayofweek(col("Full_Timestamp")))
    .withColumn("Is_Weekend", when(col("Day_Number").isin(1, 7), True).otherwise(False))
)

# --- AUDITORÍA ---
print("-" * 55)
print("✅ WRANGLING COMPLETADO")
print(f"📊 Registros listos para análisis: {df_final.count():,}")
print("-" * 55)

print("\n📋 Columnas enriquecidas — top 5:")
display(df_final.select(
    "Full_Timestamp", "Hour", "Day_Number", "Is_Weekend",
    "Global_active_power", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"
).limit(5).toPandas())

-------------------------------------------------------
✅ WRANGLING COMPLETADO


📊 Registros listos para análisis: 2,049,280
-------------------------------------------------------

📋 Columnas enriquecidas — top 5:


,Full_Timestamp,Hour,Day_Number,Is_Weekend,Global_active_power,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2006-12-16 17:24:00,17,7,True,4.216,0.0,1.0,17.0
1,2006-12-16 17:25:00,17,7,True,5.360,0.0,1.0,16.0
2,2006-12-16 17:26:00,17,7,True,5.374,0.0,2.0,17.0
3,2006-12-16 17:27:00,17,7,True,5.388,0.0,1.0,17.0
4,2006-12-16 17:28:00,17,7,True,3.666,0.0,1.0,17.0


---
## 3. Capa de análisis SQL

* **Acción:** Registrar `power_data` como vista temporal de Spark.
* **Justificación:** Esto desacopla la lógica de transformación (Python/Spark) de la lógica de negocio (SQL), permitiendo consultas limpias y legibles en todos los notebooks de hipótesis.

In [5]:
# ==============================================================================
# 2.3 SQL ANALYTICS LAYER — CATALOG REGISTRATION
# ==============================================================================
spark.catalog.dropTempView("power_data")
df_final.createOrReplaceTempView("power_data")

print("-" * 55)
print("✅ MOTOR SQL LISTO  |  Vista 'power_data' registrada")
print("🔍 Listo para consultas spark.sql()")
print("-" * 55)

-------------------------------------------------------
✅ MOTOR SQL LISTO  |  Vista 'power_data' registrada
🔍 Listo para consultas spark.sql()
-------------------------------------------------------


---
## 4. Persistencia — Exportación a Parquet

CSV → Parquet (compresión Snappy) reduce el tamaño del archivo ~80% y entrega lecturas con poda de columnas para los notebooks de hipótesis posteriores.

Particionado por `año/mes` para optimizar escaneos de rango en consultas de series temporales.

In [6]:
# ==============================================================================
# 4. PARQUET PERSISTENCE (COLUMNAR STORAGE)
# ==============================================================================
import time

start_export = time.time()

# --- RUTA DE SALIDA ---
parquet_path = os.path.join(PROJECT_ROOT, "data_storage", "work", "power_data.parquet")

# --- ESCRITURA ---
(
    df_final
    .write
    .mode("overwrite")
    .option("compression", "snappy")
    .parquet(parquet_path)
)

elapsed = time.time() - start_export

print("-" * 55)
print("💾 EXPORTACIÓN A PARQUET COMPLETADA")
print(f"   Ruta    : {parquet_path}")
print(f"   Duración: {elapsed:.2f} s")
print("-" * 55)
print("\n🚀 Siguiente paso: ejecuta cualquiera de los notebooks H1–H4.")

-------------------------------------------------------
💾 EXPORTACIÓN A PARQUET COMPLETADA
   Ruta    : /home/aitor/Documentos/Residential-Power-Audit-Spark-Pipeline/data_storage/work/power_data.parquet
   Duración: 6.00 s
-------------------------------------------------------

🚀 Siguiente paso: ejecuta cualquiera de los notebooks H1–H4.
